# Lecture 4: Context Engineering & Memory (hands-on)
**Strands Agents with Ollama · Lecture 4 of 5** · companion to `Lecture04_Context_and_Memory.pdf`

You will:
1. Inject ephemeral context with `ContextInjector`
2. Offload a huge tool result with `ContextManager` + `Offload`
3. See why `num_ctx` matters on Ollama
4. Persist and restore conversations with session managers
5. Add long-term memory with `MemoryManager`

## 0. Setup: Ollama + Strands
Same steps as `StrandsAgent_SetUp.ipynb`. Run these once per Colab session (about 2 to 3 minutes).

> **Tip:** *Runtime → Change runtime type → T4 GPU* makes Ollama much faster. CPU works too, just slower.

In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background and wait until it answers
import subprocess, time, urllib.request
subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start - check /tmp/ollama.log")

In [ ]:
!ollama pull qwen3:0.6b
# Optional, more reliable for tool use (1.4 GB):
# !ollama pull qwen3:1.7b

In [ ]:
!pip install -q "strands-agents[ollama,web-fetch]"

### Shared model helper
Every lecture uses the same `make_model()` helper. Change `MODEL_ID` here to switch models for the whole notebook.

In [ ]:
from strands.models.ollama import OllamaModel

MODEL_ID = "qwen3:0.6b"          # try "qwen3:1.7b" for better tool use

def make_model(model_id=MODEL_ID, **kwargs):
    return OllamaModel(
        host="http://127.0.0.1:11434",
        model_id=model_id,
        options={"num_ctx": 8192},   # explicit context window (see Lecture 4)
        **kwargs,
    )

import importlib.metadata as md_
print("strands-agents", md_.version("strands-agents"), "| model:", MODEL_ID)

## 1. Select: ContextInjector
The injected text augments **one** model call and is **never stored** in history.

In [ ]:
from datetime import datetime, timezone
from strands import Agent
from strands.vended_plugins.context_injector import ContextInjector

agent = Agent(
    model=make_model(),
    callback_handler=None,
    plugins=[ContextInjector(lambda context: f"<now>{datetime.now(timezone.utc).isoformat()}</now>")],
)
print(agent("What year is it right now? Answer with just the year."))
print("Stored user message:", agent.messages[0]["content"])

## 2. A tool that returns a very large result

In [ ]:
from strands import tool

@tool
def read_server_log() -> str:
    """Read the full server log file."""
    lines = [f"2026-11-14 10:{i % 60:02d} INFO request ok id={i}" for i in range(800)]
    return "\n".join(lines) + "\nERROR db timeout at 10:59"

print(len(read_server_log()), "characters")

## 3. Compress & externalize: Offload strategies
`threshold` must be **greater than 1000** (the preview size); otherwise Strands raises a `ValueError`.

In [ ]:
from strands.experimental.context_manager import ContextManager, Offload

agent = Agent(
    model=make_model(),
    tools=[read_server_log],
    callback_handler=None,
    context_manager=ContextManager(strategies=[
        Offload.truncate("tool_results").when(threshold=1500),
        Offload.summarize("*").when(utilization=0.85, preserve_recent=6),
    ]),
    system_prompt="Always use read_server_log to inspect logs.",
)
result = agent("Read the server log. Is there an ERROR line? Quote it.")
print(result)
print("Tools now available:", agent.tool_names)     # retrieve_context was added automatically
print("Input tokens:", result.metrics.accumulated_usage["inputTokens"])

In [ ]:
# What actually stayed in context for the tool result?
for m in agent.messages:
    for b in m["content"]:
        if "toolResult" in b:
            text = str(b["toolResult"]["content"])
            print(len(text), "chars kept in context")
            print(text[:400], "...")

## 4. Why `num_ctx` matters on Ollama
Compare the same question **without** offloading at a small vs a large context window. With a small window, Ollama silently drops the oldest tokens.

In [ ]:
def no_offload_trial(num_ctx):
    model = OllamaModel(host="http://127.0.0.1:11434", model_id=MODEL_ID, options={"num_ctx": num_ctx})
    a = Agent(model=model, tools=[read_server_log], callback_handler=None,
              system_prompt="Always use read_server_log to inspect logs.")
    r = a("Read the server log. Is there an ERROR line? Quote it.")
    print(f"num_ctx={num_ctx:<6} input_tokens={r.metrics.accumulated_usage['inputTokens']:<6} -> {str(r).strip()[:120]}")

no_offload_trial(2048)
no_offload_trial(16384)

## 5. `context_manager="auto"`: the one-line default

In [ ]:
auto_agent = Agent(model=make_model(), tools=[read_server_log], context_manager="auto", callback_handler=None,
                   system_prompt="Always use read_server_log to inspect logs.")
print(auto_agent("Read the server log and report any ERROR lines."))
print(auto_agent.tool_names)

## 6. Persist a conversation: FileSessionManager
We create a **brand-new agent** with the same `session_id` to simulate a restart.

In [ ]:
from strands.session.file_session_manager import FileSessionManager

sm = FileSessionManager(session_id="customer-001", storage_dir="./sessions")
agent = Agent(model=make_model(), session_manager=sm, callback_handler=None)
print(agent("Hi, my name is Mai and my order number is A-1042. Reply OK."))

In [ ]:
# --- "restart" ---
sm2 = FileSessionManager(session_id="customer-001", storage_dir="./sessions")
agent2 = Agent(model=make_model(), session_manager=sm2, callback_handler=None)
print(len(agent2.messages), "messages restored:")
for m in agent2.messages:
    print(" ", m["role"], "|", m["content"][0].get("text", "")[:70])
print("\nAnswer:", agent2("What is my order number?"))

In [ ]:
!find sessions -name "*.json" | sort

> If the history was restored but the answer is wrong, that's a **model** problem, not a storage problem. Try `qwen3:1.7b`.

## 7. SnapshotSessionManager + long-term memory

In [ ]:
from strands.session import SnapshotSessionManager
from strands.storage import LocalFileStorage
from strands.memory import MemoryManager
from strands.vended_memory_stores.test_memory_store import TestMemoryStore

session = SnapshotSessionManager(session_id="customer-002", storage=LocalFileStorage("./sessions"))
store = TestMemoryStore(name="customer-service")

agent = Agent(
    model=make_model(),
    callback_handler=None,
    context_manager="auto",
    session_manager=session,
    memory_manager=MemoryManager(stores=[store]),
)
print(agent.tool_names)
print(agent("Remember: the project codename is BLUEBIRD. Reply OK."))

In [ ]:
session2 = SnapshotSessionManager(session_id="customer-002", storage=LocalFileStorage("./sessions"))
agent3 = Agent(model=make_model(), session_manager=session2, callback_handler=None)
print(len(agent3.messages), "messages restored")
print(agent3("What is the project codename?"))

## 8. Your turn
**Exercise 1:** Write a `ContextInjector` that injects a fake user profile (name, plan tier, preferred language). Ask a question that depends on it and confirm it's not in `agent.messages`.

**Exercise 2:** Build a cheaper-summarizer setup: `qwen3:1.7b` as the main model and `qwen3:0.6b` for `Offload.summarize` with a domain-specific prompt.

**Exercise 3 (stretch):** Hold a 10-turn conversation with `num_ctx=2048` and `Offload.summarize("*").when(utilization=0.5, preserve_recent=4)`. Then print `agent.messages` and read the summary that replaced old turns.

In [ ]:
# Your experiments here